In [ ]:
from google.colab import drive
import pandas as pd
# Mount your Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/ĐACN1/Medicalpremium_not2.csv")
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1008 entries, 0 to 1007
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      1007 non-null   float64
 1   Diabetes                 1007 non-null   float64
 2   BloodPressureProblems    1007 non-null   float64
 3   AnyTransplants           1007 non-null   object 
 4   AnyChronicDiseases       1007 non-null   object 
 5   Height                   1007 non-null   float64
 6   Weight                   1006 non-null   float64
 7   KnownAllergies           1006 non-null   object 
 8   HistoryOfCancerInFamily  1006 non-null   object 
 9   NumberOfMajorSurgeries   1005 non-null   object 
 10  PremiumPrice             1006 non-null   float64
dtypes: float64(6), object(5)
memory usage: 86.8+ KB


,Age,Diabetes,BloodPressureProblems,AnyTransplants,AnyChronicDiseases,Height,Weight,KnownAllergies,HistoryOfCancerInFamily,NumberOfMajorSurgeries,PremiumPrice
0,45.0,0.0,0.0,0,0,155.0,57.0,0,0,0,25000.0
1,60.0,1.0,0.0,0,0,180.0,73.0,0,0,0,29000.0
2,36.0,1.0,1.0,0,0,158.0,59.0,0,0,1,23000.0
3,52.0,1.0,1.0,0,1,183.0,93.0,0,0,2,28000.0
4,38.0,0.0,0.0,0,1,166.0,88.0,0,0,1,23000.0


In [ ]:
df.shape

(1008, 11)

In [ ]:
df.isnull().sum()

Age                        1
Diabetes                   1
BloodPressureProblems      1
AnyTransplants             1
AnyChronicDiseases         1
Height                     1
Weight                     2
KnownAllergies             2
HistoryOfCancerInFamily    2
NumberOfMajorSurgeries     3
PremiumPrice               2
dtype: int64

In [ ]:
# Tạo một DataFrame mới với các hàng lỗi
new_rows = pd.DataFrame(
    {
        "Age": [-10, 150, 0, 35],
        "Diabetes": ["yes", 2, "Unknown", "no"],
        "BloodPressureProblems": ["yes", "no", 3, "no"],
        "AnyTransplants": ["yes", "no", "Unknown", "no"],
        "AnyChronicDiseases": ["yes", "no", "Unknown", "no"],
        "Height": [-10, 300, 0, 175],
        "Weight": [-10, 300, 0, 85],
        "KnownAllergies": ["yes", "no", "Unknown", "no"],
        "HistoryOfCancerInFamily": ["yes", "no", "Unknown", "no"],
        "NumberOfMajorSurgeries": [-1, 3, "Unknown", 2],
        "PremiumPrice": [25000, 32000, "Unknown", 28000],
    }
)

# Nối DataFrame mới vào DataFrame hiện có
df_with_errors = pd.concat([df, new_rows], ignore_index=True)

# In DataFrame kết quả
print(df_with_errors.tail().to_markdown(index=False, numalign="left", stralign="left"))

| Age   | Diabetes   | BloodPressureProblems   | AnyTransplants   | AnyChronicDiseases   | Height   | Weight   | KnownAllergies   | HistoryOfCancerInFamily   | NumberOfMajorSurgeries   | PremiumPrice   |
|:------|:-----------|:------------------------|:-----------------|:---------------------|:---------|:---------|:-----------------|:--------------------------|:-------------------------|:---------------|
| 21    | 2.0        | 3.0                     | yes              | no                   | 158      | 75       | 1                | 0                         | 1                        | 15000.0        |
| -10   | yes        | yes                     | yes              | yes                  | -10      | -10      | yes              | yes                       | -1                       | 25000          |
| 150   | 2          | no                      | no               | no                   | 300      | 300      | no               | no                        | 3                       

In [ ]:
# Trộn ngẫu nhiên thứ tự các hàng trong DataFrame
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Lưu DataFrame đã trộn vào một file CSV mới có tên là "Medicalpremium_shuffled.csv"
df_shuffled.to_csv('Medicalpremium_shuffled.csv', index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def preprocess_data(df):
    # Chuyển đổi các giá trị "yes" và "no" thành 1 và 0
    yes_no_columns = ['Diabetes', 'BloodPressureProblems', 'AnyTransplants', 'AnyChronicDiseases', 'KnownAllergies', 'HistoryOfCancerInFamily']
    df[yes_no_columns] = df[yes_no_columns].replace({'yes': 1, 'no': 0})
    # Loại bỏ các hàng có dữ liệu bị thiếu
    df = df.dropna()
    # Chuyển đổi tất cả các cột sang dạng số, các giá trị không thể chuyển đổi sẽ trở thành NaN
    df = df.apply(pd.to_numeric, errors='coerce')
    # Loại bỏ các hàng có giá trị không phải là số trong các cột yes_no_columns (tức là các giá trị NaN sau khi chuyển đổi)
    df = df.dropna(subset=yes_no_columns)
    # Xác định các giá trị hợp lệ
    valid_values = [0, 1]
    # Loại bỏ các hàng mà các cột được chỉ định có giá trị khác 0 hoặc 1
    for column in yes_no_columns:
        df = df[df[column].isin(valid_values)]
    # Loại bỏ các hàng trùng lặp
    df = df.drop_duplicates()
    # Loại bỏ các hàng có cột Age, Height, Weight có giá trị bằng 0 hoặc vượt quá giới hạn
    df = df[(df['Age'] > 0) & (df['Age'] <= 123)]
    df = df[(df['Height'] > 0) & (df['Height'] <= 272)]
    df = df[(df['Weight'] > 0) & (df['Weight'] <= 290)]
    return df

In [ ]:
df = preprocess_data(df)

In [ ]:
df.shape

(987, 11)

In [ ]:
# Specify the path to the output CSV file
#output_csv_file_path = '/content/drive/My Drive/ĐACN1/processed_data.csv'

# Save the preprocessed data to a CSV file
#data.to_csv(output_csv_file_path, index=False)